# Evaluate molecular feasibility

**Purpose:** Evaluate molecular feasibility.

**Before you start:** Saved molecular model and matching ZINC data. Use the Python environment prepared by [setup](../setup.ipynb).

**Results:** Decode traces, score comparisons, and interpolation plots.

Run the cells in order, reviewing the configuration before starting the main work. Data stays under `notebooks/datasets`; models and outputs use the project’s artifact folders.


Load the molecule and feasibility analysis tools.


In [ ]:
print('Load the molecule and feasibility analysis tools.')
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

import random
from pathlib import Path

import numpy as np
import pandas as pd


from conditional_node_field_graph_generator.notebooks import configure_notebook

globals().update(configure_notebook(require_nsppk=True, print_torch=True))

from conditional_node_field_graph_generator.extensions.demo import (
    collect_oracle_trace_rows,
    oracle_trace_frame,
    parse_oracle_trace_title,
    show_molecules,
)
from conditional_node_field_graph_generator.extensions.demo.pipeline import prepare_zinc_data_split
from abstractgraph_graphicalizer.chem import (
    draw_molecules,
)
from conditional_node_field_graph_generator.persistence import load_graph_generator


## Helpers

The helpers below keep the oracle comparison logic explicit and reproducible.


Prepare the helpers used to trace and compare decoding.


In [ ]:
print('Prepare the helpers used to trace and compare decoding.')
def _set_all_seeds(seed):
    random.seed(int(seed))
    np.random.seed(int(seed))


def _fmt_score(value):
    return '-inf' if value is None else f'{value:.3f}'


def _print_oracle_trace_rows(trace_rows):
    for row in trace_rows:
        print(
            f"Oracle {row['phase']} iteration={row['iteration']} "
            f"violating_node_sets={row['violating_node_sets']} "
            f"violating_edge_sets={row['violating_edge_sets']} "
            f"new_structural_cuts={row['new_structural_cuts']} "
            f"accepted_structural_cuts={row['accepted_structural_cuts']}"
        )
        if row.get('log_total') is not None:
            print(
                '  proposal log-score: '
                f"total={row['log_total']:.3f} | "
                f"edge={0.0 if row.get('log_edge') is None else row['log_edge']:.3f} | "
                f"node={0.0 if row.get('log_node') is None else row['log_node']:.3f} | "
                f"edge_label={0.0 if row.get('log_edge_label') is None else row['log_edge_label']:.3f}"
            )
            print(
                '  running best: '
                f"best_total={_fmt_score(row.get('best_log_total'))} | "
                f"best_feasible={_fmt_score(row.get('best_feasible_log_total'))}"
            )


def show_oracle_decode_trace(
    graph_generator,
    seed_graph,
    *,
    feasibility_effort=2,
    seed=7,
):
    show_molecules([seed_graph], n=1, title='Conditioning molecule')
    if getattr(graph_generator, 'feasibility_estimator', None) is None:
        print('Feasibility estimator unavailable; oracle trace cannot be shown.')
        return {'conditioning_graph': seed_graph, 'oracle_on': [], 'trace': []}
    if graph_generator.graph_decoder.verbose < 4 or graph_generator.graph_decoder.n_jobs != 1:
        print('Set DECODER_VERBOSE >= 4 and DECODER_N_JOBS = 1 to display inline oracle phases and per-iteration decoder diagnostics.')

    print(
        'Oracle ON trace for the same conditioning graph '
        f'(effort={feasibility_effort}, '
        f'max_oracle_iterations={graph_generator.max_oracle_iterations}).'
    )
    print('The selected effort policy is used for one deterministic decode trajectory.')
    print('If the initial connectivity-constrained seed solve stalls, the oracle path now retries the seed solve with connectivity disabled before continuing.')
    print('Rows are emitted separately for the joint-label phase, structural edge-set phase, and feasibility checks.')
    _set_all_seeds(seed)
    with collect_oracle_trace_rows() as trace_rows:
        oracle_on = graph_generator.sample_conditioned_on_random(
            [seed_graph],
            n_samples=1,
            feasibility_effort=feasibility_effort,
            feasibility_filter='strict',
        )
    trace_frame = oracle_trace_frame(trace_rows)
    if not trace_frame.empty:
        import matplotlib.pyplot as plt
        _print_oracle_trace_rows(trace_rows)
        display(trace_frame)
        fig, ax = plt.subplots(figsize=(18, 4))
        ax_cuts = ax.twinx()
        ax_log_total = ax.twinx()
        ax_log_total.spines['right'].set_position(('outward', 60))
        x = np.arange(len(trace_frame), dtype=float)
        width = 0.26
        bars_node = ax.bar(
            x - width,
            trace_frame['violating_node_sets'],
            width=width,
            label='violating_node_sets',
        )
        bars_edge = ax.bar(
            x,
            trace_frame['violating_edge_sets'],
            width=width,
            label='violating_edge_sets',
        )
        bars_cuts = ax_cuts.bar(
            x + width,
            trace_frame['accepted_structural_cuts'],
            width=width,
            label='accepted_structural_cuts',
            color='tab:green',
        )
        log_total = pd.to_numeric(trace_frame.get('log_total'), errors='coerce')
        line_log_total, = ax_log_total.plot(
            x,
            log_total,
            color='tab:red',
            linewidth=2.0,
            marker='o',
            markersize=4,
            label='log_total',
        )
        for label_axis, bar_group in ((ax, (bars_node, bars_edge)), (ax_cuts, (bars_cuts,))):
            for group in bar_group:
                for bar in group:
                    height = bar.get_height()
                    label_axis.text(
                        bar.get_x() + bar.get_width() / 2.0,
                        height + 0.15,
                        f'{int(height)}',
                        ha='center',
                        va='bottom',
                        fontsize=9,
                    )
        for xi, yi in zip(x, log_total):
            if pd.isna(yi):
                continue
            ax_log_total.text(
                xi,
                yi,
                f'{yi:.2f}',
                color='tab:red',
                fontsize=8,
                ha='center',
                va='bottom',
            )
        ax.set_title('Oracle trace metrics by iteration')
        ax.set_xlabel('iteration')
        ax.set_ylabel('violations')
        ax_cuts.set_ylabel('accepted structural cuts')
        ax_log_total.set_ylabel('log_total')
        ax.tick_params(axis='y', labelcolor='black')
        ax_cuts.tick_params(axis='y', labelcolor='tab:green')
        ax_log_total.tick_params(axis='y', labelcolor='tab:red')
        ax.set_xticks(x)
        ax.set_xticklabels(trace_frame['iteration'].astype(int).tolist())
        ax.grid(alpha=0.3)
        handles = [bars_node, bars_edge, bars_cuts, line_log_total]
        labels = [artist.get_label() for artist in handles]
        ax.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, -0.14), ncol=4, frameon=True)
        plt.tight_layout(rect=(0, 0.08, 1, 1))
        plt.show()
    show_molecules(oracle_on, n=1, title='Final oracle-guided decoded sample')
    return {'conditioning_graph': seed_graph, 'oracle_on': oracle_on, 'trace': trace_rows, 'trace_frame': trace_frame}


def compare_oracle_score(
    graph_generator,
    *,
    n_samples=24,
    seed=11,
):
    """Compare fixed effort profiles while keeping the random seed identical."""
    original_decoder_verbose = graph_generator.graph_decoder.verbose
    graph_generator.graph_decoder.verbose = 0
    try:
        _set_all_seeds(seed)
        score_without = graph_generator.score_feasible_rate(
            n_samples=n_samples,
            feasibility_effort=1,
            verbose=False,
        )
        _set_all_seeds(seed)
        score_with = graph_generator.score_feasible_rate(
            n_samples=n_samples,
            feasibility_effort=2,
            verbose=False,
        )
    finally:
        graph_generator.graph_decoder.verbose = original_decoder_verbose

    final_counts = pd.DataFrame(
        [
            {
                'accepted_slots': score_without['accepted_slots'],
                'n_samples': score_without['n_samples'],
                'generated_candidates': score_without['generated_candidates'],
                'feasible_candidates': score_without['feasible_candidates'],
            },
            {
                'accepted_slots': score_with['accepted_slots'],
                'n_samples': score_with['n_samples'],
                'generated_candidates': score_with['generated_candidates'],
                'feasible_candidates': score_with['feasible_candidates'],
            },
        ],
        index=['effort_1', 'effort_2_oracle'],
    )
    print(final_counts.to_string())
    return final_counts


def compare_oracle_interpolation(
    graph_generator,
    graph_a,
    graph_b,
    *,
    k=5,
    off_effort=1,
    on_effort=2,
):
    """Compare interpolation under ILP feasibility and oracle-guided feasibility."""
    original_decoder_verbose = graph_generator.graph_decoder.verbose
    graph_generator.graph_decoder.verbose = 0
    try:
        oracle_off = graph_generator.interpolate(
            graph_a,
            graph_b,
            k=k,
            feasibility_effort=off_effort,
            feasibility_filter='strict',
        )
        oracle_on = graph_generator.interpolate(
            graph_a,
            graph_b,
            k=k,
            feasibility_effort=on_effort,
            feasibility_filter='strict',
        )
    finally:
        graph_generator.graph_decoder.verbose = original_decoder_verbose

    summary = pd.DataFrame(
        {
            'step': oracle_on['summary']['step'],
            't': oracle_on['summary']['t'],
            'oracle_off_decoded': oracle_off['summary']['decoded'],
            'oracle_on_decoded': oracle_on['summary']['decoded'],
        }
    )
    show_molecules(oracle_off['generated_graphs'], n=k, title='Interpolation with effort 1')
    show_molecules(oracle_on['generated_graphs'], n=k, title='Interpolation with effort 2 oracle')
    return {'summary': summary, 'oracle_off': oracle_off, 'oracle_on': oracle_on}


def load_oracle_study_state():
    zinc_data = prepare_zinc_data_split(
        dataset_dir=ZINC_DATA_ROOT,
        num_examples=MAX_MOLECULES,
        min_size=MIN_NODE_COUNT,
        max_size=MAX_NODE_COUNT,
        test_size=max(32, min(256, MAX_MOLECULES // 10)),
        random_state=RANDOM_SEED,
    )
    graph_generator = load_graph_generator(MODEL_FILENAME, model_dir=SAVED_GENERATOR_ROOT)
    if graph_generator.feasibility_estimator is None:
        print('Feasibility support is unavailable; effort-based filtering will return unfiltered samples.')
    else:
        graph_generator.feasibility_estimator.set_parallel(False)
    graph_generator.max_oracle_iterations = MAX_ORACLE_ITERATIONS
    graph_generator.oracle_use_node_label_cuts = True
    graph_generator.oracle_use_edge_label_cuts = True
    graph_generator.feasibility_failure_mode = FEASIBILITY_FAILURE_MODE
    graph_generator.verbose = DECODER_VERBOSE
    graph_generator.graph_decoder.verbose = DECODER_VERBOSE
    graph_generator.graph_decoder.n_jobs = DECODER_N_JOBS
    graph_generator.graph_decoder.diagnostic_graph_renderer = DECODER_GRAPH_RENDERER
    return zinc_data, graph_generator


## Configuration

Set the uploaded model filename or an absolute path to the saved generator snapshot.


Choose a saved model, input molecules, and feasibility settings.


In [ ]:
print('Choose a saved model, input molecules, and feasibility settings.')
MODEL_FILENAME = 'zinc15-streaming-d64-s0-5-w1024-b32-e256.pkl'  # Replace with your saved model filename if needed.
ZINC_DATA_ROOT = NOTEBOOK_DATA_ROOT / 'zinc'
MAX_MOLECULES = 64
MIN_NODE_COUNT = 10
MAX_NODE_COUNT = 15
N_SEEDS = 2
TRACE_SEED_INDEX = 0
N_SCORE_SAMPLES = 4
FEASIBILITY_EFFORT = 2
FEASIBILITY_FILTER = 'strict'
ORACLE_OFF_EFFORT = 1
ORACLE_ON_EFFORT = 2
MAX_FEASIBILITY_ATTEMPTS = 9
FEASIBILITY_FAILURE_MODE = 'return_partial'
MAX_ORACLE_ITERATIONS = 60
DECODER_VERBOSE = 4  # Raise to 4 to show per-decode diagnostic plots during oracle runs.
DECODER_N_JOBS = 1  # Keep this at 1 when DECODER_VERBOSE >= 4 so plots can be shown.
DECODER_GRAPH_RENDERER = draw_molecules  # Optional callback used for per-decode graph rendering.
RANDOM_SEED = 29


## Load Cached ZINC Slice And Uploaded Generator


Load the selected ZINC data and saved model.


In [ ]:
print('Load the selected ZINC data and saved model.')
zinc_data, graph_generator = load_oracle_study_state()
globals().update(zinc_data)
print(f"Loaded {len(graphs)} ZINC graphs for oracle study from {manifest['dataset_name']}.csv.")
if DECODER_VERBOSE >= 4 and DECODER_N_JOBS != 1:
    print('Set DECODER_N_JOBS = 1 to display decoder diagnostic plots.')
print('Decoder verbose level =', graph_generator.graph_decoder.verbose)
print('Decoder n_jobs =', graph_generator.graph_decoder.n_jobs)
print('Decoder graph renderer =', getattr(graph_generator.graph_decoder.diagnostic_graph_renderer, '__name__', graph_generator.graph_decoder.diagnostic_graph_renderer))
print('max_oracle_iterations =', graph_generator.max_oracle_iterations)
print('oracle_use_node_label_cuts =', graph_generator.oracle_use_node_label_cuts)
print('oracle_use_edge_label_cuts =', graph_generator.oracle_use_edge_label_cuts)
print('Configured scientific policy = effort', FEASIBILITY_EFFORT, 'filter', FEASIBILITY_FILTER)


## Seed Molecules

Pick a small deterministic set of seed molecules that will be reused across the comparisons.


Choose and display the molecules used to condition decoding.


In [ ]:
print('Choose and display the molecules used to condition decoding.')
rng = np.random.default_rng()
seed_indices = rng.choice(len(graphs), size=N_SEEDS, replace=False)
seed_graphs = [graphs[int(idx)] for idx in seed_indices]
print('Selected seed indices:', seed_indices.tolist())
show_molecules(seed_graphs, n=N_SEEDS, title='Seed molecules for oracle study')
trace_seed_graph = seed_graphs[int(TRACE_SEED_INDEX)]
print('Trace seed index within seed_graphs =', int(TRACE_SEED_INDEX))
_ = show_molecules([trace_seed_graph], n=1, title='Seed used for oracle phase trace')


## Oracle Phase Trace

Decode one conditioning molecule with the oracle enabled so the per-iteration diagnostic panels show how accepted violating motifs accumulate across phases. Feasibility filtering is disabled in this trace cell to avoid rendering large candidate batches.


**Oracle strategy summary.** The decoder keeps structural edge-set cuts as the hard optimization mechanism. Node-label and edge-label changes are handled as soft follow-up repair proposals after each structural update, and a relabeling is accepted only if it improves the full oracle state rather than a label-only phase score.


Trace the decoding process to see how feasibility constraints are applied.


In [ ]:
print('Trace the decoding process to see how feasibility constraints are applied.')
oracle_phase_trace = show_oracle_decode_trace(
    graph_generator,
    trace_seed_graph,
    feasibility_effort=FEASIBILITY_EFFORT,
    seed=RANDOM_SEED,
)
_ = show_molecules([trace_seed_graph], n=1, title='Seed used for oracle phase trace')


## Feasible-Rate Score Comparison

Keep the retry budget fixed and compare how often the full decode pipeline yields feasible candidates with and without the oracle.


Compare decoding scores with and without the feasibility oracle.


In [ ]:
print('Compare decoding scores with and without the feasibility oracle.')
oracle_score_comparison = compare_oracle_score(
    graph_generator,
    n_samples=N_SCORE_SAMPLES,
    seed=RANDOM_SEED + 1,
)


## Interpolation Comparison

Inspect whether the oracle changes which intermediate molecules successfully decode along the same interpolation path.


Compare interpolation results under the two feasibility policies.


In [ ]:
print('Compare interpolation results under the two feasibility policies.')
endpoint_a, endpoint_b = seed_graphs[0], seed_graphs[1]
oracle_interpolation_comparison = compare_oracle_interpolation(
    graph_generator,
    endpoint_a,
    endpoint_b,
    k=5,
    off_effort=ORACLE_OFF_EFFORT,
    on_effort=ORACLE_ON_EFFORT,
)
